# Phase 3+4 integration — 5-seed sweep (Colab)

Runs `experiments/19_phase34_integrated.py` for seeds {17, 11, 23, 1, 2} on CUDA.

**Prereqs (do once, manually):**
1. Upload `phase3c_codebook_reconstruction.pt` (64MB, from `reports/phase3c_reconstruction/`) to Google Drive at `MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt`. The `.pt` files are `.gitignored` so the repo clone won't include them.
2. (Optional) Set Runtime → GPU (T4 is free; A100 if Pro). Roughly 10 min/seed on T4, ~2 min/seed on A100.
3. Run cells top to bottom.

Results are copied back to `MyDrive/neuro-ai/results/` so you don't lose them when the Colab VM dies.

In [ ]:
# 1. Clone the repo (public or private — for private, configure a token first).
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI

In [ ]:
# 2. Mount Drive and copy the codebook into place.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt'
dst_dir = 'reports/phase3c_reconstruction'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, f'{dst_dir}/phase3c_codebook_reconstruction.pt')
!ls -lh {dst_dir}/phase3c_codebook_reconstruction.pt

In [ ]:
# 3. Install deps. Colab ships torch+CUDA; we just need torchhd and datasets.
!pip install -q torchhd datasets

In [ ]:
# 4. Sanity check: GPU present, FHRR + codebook load.
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

import sys; sys.path.insert(0, 'src')
from energy_memory.phase2.persistence import load_codebook
cb = load_codebook('reports/phase3c_reconstruction/phase3c_codebook_reconstruction.pt', device='cuda')
print('codebook:', cb.shape, cb.dtype, cb.device)

In [ ]:
# 5. Run 5-seed sweep sequentially. Logs and per-seed result dirs live under reports/.
import subprocess, os, time
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
log_root = Path('reports/phase34_integrated_hebbian_5seed_colab')
log_root.mkdir(parents=True, exist_ok=True)

for seed in [17, 11, 23, 1, 2]:
    out_dir = f'reports/phase34_integrated_hebbian_seed{seed}'
    log_path = log_root / f'seed{seed}.log'
    print(f'\n=== seed {seed} -> {out_dir} ===  ({time.strftime("%H:%M:%S")})')
    with open(log_path, 'w') as logf:
        proc = subprocess.run(
            ['python', 'experiments/19_phase34_integrated.py',
             '--updater-kind', 'hebbian',
             '--seed', str(seed),
             '--device', 'cuda',
             '--output-dir', out_dir],
            stdout=logf, stderr=subprocess.STDOUT,
        )
    print(f'    exit={proc.returncode}  log={log_path}')
    !tail -6 {log_path}

In [ ]:
# 6. Copy results back to Drive so the VM dying doesn't lose them.
import shutil, os
dst = '/content/drive/MyDrive/neuro-ai/results'
os.makedirs(dst, exist_ok=True)
for seed in [17, 11, 23, 1, 2]:
    src = f'reports/phase34_integrated_hebbian_seed{seed}'
    if os.path.isdir(src):
        shutil.copytree(src, f'{dst}/phase34_integrated_hebbian_seed{seed}', dirs_exist_ok=True)
shutil.copytree('reports/phase34_integrated_hebbian_5seed_colab', f'{dst}/phase34_integrated_hebbian_5seed_colab', dirs_exist_ok=True)
print('results in', dst)
!ls {dst}

In [ ]:
# 7. Quick summary table across seeds — final-checkpoint top1/cap_t05 per condition.
import json
from pathlib import Path

print(f'{"seed":>4} {"cond":>16} {"top1":>8} {"topk":>8} {"cap_t05":>8} {"cues":>6}')
for seed in [17, 11, 23, 1, 2]:
    p = Path(f'reports/phase34_integrated_hebbian_seed{seed}/phase34_results.json')
    if not p.exists(): continue
    d = json.loads(p.read_text())
    for cond in ('baseline_static', 'phase3_reencode', 'phase3_phase4'):
        final = d['results'][cond][-1]
        print(f'{seed:>4} {cond:>16} {final["top1"]:>8.3f} {final["topk"]:>8.3f} {final["cap_t_05"]:>8.3f} {final["cues_seen"]:>6}')